# Módulo 1 — Estruturas de Controle

Estruturas de controle determinam o **fluxo de execução** do programa. Neste notebook aplicamos if/elif/else, for, while e list comprehensions a regras reais do setor elétrico.

## O que vamos ver

1. `if / elif / else` — classificação tarifária
2. `for` — iteração sobre leituras de medidores
3. `while` — validação iterativa de dados
4. List comprehensions — filtragem eficiente
5. Prática: classificar consumidores por faixa de consumo (ANEEL)

---

> **Contexto:** A ANEEL define faixas de consumo para consumidores residenciais da subclasse Baixa Renda. Mas qualquer consumidor pode ser classificado em faixas para análise de portfólio.

## 1. if / elif / else — Classificação Tarifária

A modalidade tarifária de um consumidor define como ele é cobrado. A classificação depende de tensão, grupo e características do ponto de entrega.

In [ ]:
# Classificação de grupo tarifário por tensão de fornecimento
def classificar_grupo_tarifario(tensao_kv: float) -> str:
    """
    Classifica o grupo tarifário com base na tensão de fornecimento.
    Grupos do PRODIST/ANEEL:
      A1 >= 230 kV
      A2:  88 kV a 138 kV
      A3:  69 kV
      A3a: 30 kV a 44 kV
      A4:  2,3 kV a 25 kV
      AS:  Subterrâneo (tensão equivalente ao A4)
      B:   < 1 kV (baixa tensão)
    """
    if tensao_kv >= 230:
        return "A1"
    elif tensao_kv >= 88:
        return "A2"
    elif tensao_kv == 69:
        return "A3"
    elif tensao_kv >= 30:
        return "A3a"
    elif tensao_kv >= 2.3:
        return "A4"
    elif tensao_kv < 1:
        return "B"
    else:
        return "Indefinido"

# Testes
tensoes = [0.22, 0.38, 13.8, 34.5, 69.0, 138.0, 345.0]
for t in tensoes:
    grupo = classificar_grupo_tarifario(t)
    print(f"Tensão: {t:>7} kV → Grupo: {grupo}")

In [ ]:
# Classificação de subclasse B por tipo de consumidor
def classificar_subclasse_b(cod_classe: str, flg_baixa_renda: bool = False) -> str:
    """
    Retorna a modalidade tarifária para grupo B (baixa tensão).
    B1 - Residencial
    B2 - Rural
    B3 - Demais classes (Comercial, Industrial BT, Serviço Público)
    B4 - Iluminação Pública
    """
    classe = cod_classe.upper().strip()
    
    if classe == "RESIDENCIAL":
        if flg_baixa_renda:
            return "B1 - Residencial Baixa Renda"
        return "B1 - Residencial"
    elif classe == "RURAL":
        return "B2 - Rural"
    elif classe == "ILUMINACAO PUBLICA":
        return "B4 - Iluminação Pública"
    elif classe in ["COMERCIAL", "INDUSTRIAL", "PODER PUBLICO", "SERVICO PUBLICO"]:
        return "B3 - Demais Classes"
    else:
        return f"Classe desconhecida: {cod_classe}"

# Testes
classes_teste = [
    ("RESIDENCIAL", False),
    ("RESIDENCIAL", True),
    ("COMERCIAL", False),
    ("INDUSTRIAL", False),
    ("RURAL", False),
    ("ILUMINACAO PUBLICA", False),
    ("PODER PUBLICO", False),
]

for classe, baixa_renda in classes_teste:
    subclasse = classificar_subclasse_b(classe, baixa_renda)
    print(f"{classe:<20} (baixa renda={baixa_renda}) → {subclasse}")

## 2. for — Iteração sobre Leituras de Medidores

O laço `for` é essencial para processar séries temporais de medições.

In [ ]:
# Histórico de leituras mensais (kWh) com valores ausentes
leituras = [
    {"mes": "Jan/25", "leitura": 310},
    {"mes": "Fev/25", "leitura": 325},
    {"mes": "Mar/25", "leitura": None},   # leitura ausente
    {"mes": "Abr/25", "leitura": 298},
    {"mes": "Mai/25", "leitura": 315},
    {"mes": "Jun/25", "leitura": -5},     # valor inválido
    {"mes": "Jul/25", "leitura": 401},
    {"mes": "Ago/25", "leitura": 389},
]

# Processar leituras: separar válidas, ausentes e inválidas
leituras_validas = []
problemas = []

for item in leituras:
    mes = item["mes"]
    valor = item["leitura"]
    
    if valor is None:
        problemas.append({"mes": mes, "tipo": "AUSENTE", "valor": None})
    elif valor < 0:
        problemas.append({"mes": mes, "tipo": "INVALIDO", "valor": valor})
    else:
        leituras_validas.append(valor)

# Relatório
print(f"Total de meses: {len(leituras)}")
print(f"Leituras válidas: {len(leituras_validas)}")
print(f"Problemas encontrados: {len(problemas)}")
print()

for p in problemas:
    print(f"  {p['mes']}: {p['tipo']} (valor={p['valor']})")

if leituras_validas:
    media = sum(leituras_validas) / len(leituras_validas)
    print(f"\nMédia das leituras válidas: {media:.1f} kWh")

In [ ]:
# Usando enumerate para iterar com índice
print("Análise mês a mês:")
print(f"{'#':>3} | {'Mês':<8} | {'Leitura':>10} | {'Status':<12} | {'Var. anterior':>14}")
print("-" * 60)

leitura_anterior = None

for i, item in enumerate(leituras):
    valor = item["leitura"]
    
    if valor is None:
        status = "SEM LEITURA"
        variacao = "—"
    elif valor < 0:
        status = "INVÁLIDO"
        variacao = "—"
    else:
        status = "OK"
        if leitura_anterior is not None:
            var = ((valor - leitura_anterior) / leitura_anterior) * 100
            variacao = f"{var:+.1f}%"
        else:
            variacao = "—"
        leitura_anterior = valor
    
    valor_str = f"{valor} kWh" if valor is not None else "N/A"
    print(f"{i+1:>3} | {item['mes']:<8} | {valor_str:>10} | {status:<12} | {variacao:>14}")

## 3. while — Validação Iterativa de Dados

O `while` é útil quando não sabemos quantas iterações serão necessárias — como em processos de limpeza de dados.

In [ ]:
import re

def limpar_cep(cep: str) -> str:
    """Remove caracteres não numéricos do CEP."""
    return re.sub(r'\D', '', cep)

def validar_cep(cep: str) -> bool:
    """Verifica se o CEP tem 8 dígitos."""
    cep_limpo = limpar_cep(cep)
    return len(cep_limpo) == 8

# Simula uma fila de CEPs para validar
fila_ceps = [
    "01310-100",   # válido
    "0131010",     # inválido (7 dígitos)
    "01310-1000",  # inválido (9 dígitos)
    "13.420-000",  # válido (com ponto)
    "ABC-123",     # inválido
    "20040-020",   # válido
]

# Processar fila com while
indice = 0
validos = []
invalidos = []

while indice < len(fila_ceps):
    cep = fila_ceps[indice]
    cep_limpo = limpar_cep(cep)
    
    if validar_cep(cep):
        validos.append(cep_limpo)
        status = "✓ válido"
    else:
        invalidos.append(cep)
        status = "✗ inválido"
    
    print(f"CEP '{cep}' → {cep_limpo} — {status}")
    indice += 1

print(f"\nVálidos: {validos}")
print(f"Inválidos: {invalidos}")

## 4. List Comprehensions — Filtragem Eficiente

List comprehensions são a forma pythônica de transformar e filtrar listas em uma única expressão.

In [ ]:
# Base de consumidores para exercícios
consumidores = [
    {"cod": 1001, "nome": "João Silva",      "classe": "RESIDENCIAL", "consumo": 245.0, "ativo": True},
    {"cod": 1002, "nome": "Farmácia Saúde",  "classe": "COMERCIAL",   "consumo": 1820.5, "ativo": True},
    {"cod": 1003, "nome": "Ana Oliveira",    "classe": "RESIDENCIAL", "consumo": 89.3,  "ativo": True},
    {"cod": 1004, "nome": "Metalúrgica X",   "classe": "INDUSTRIAL",  "consumo": 85000.0, "ativo": True},
    {"cod": 1005, "nome": "Sítio Esperança","classe": "RURAL",        "consumo": 310.2, "ativo": False},
    {"cod": 1006, "nome": "Padaria Bom Dia","classe": "COMERCIAL",    "consumo": 2340.0, "ativo": True},
    {"cod": 1007, "nome": "Pedro Santos",   "classe": "RESIDENCIAL", "consumo": 1850.0, "ativo": True},
    {"cod": 1008, "nome": "Escola Estadual","classe": "PODER PUBLICO","consumo": 4200.0, "ativo": True},
]

# 1. Todos os consumidores ativos
ativos = [c for c in consumidores if c["ativo"]]
print(f"Ativos: {len(ativos)} de {len(consumidores)}")

# 2. Apenas residenciais com consumo > 500 kWh
residenciais_altos = [
    c for c in consumidores
    if c["classe"] == "RESIDENCIAL" and c["consumo"] > 500
]
print(f"\nResidenciais com consumo > 500 kWh:")
for c in residenciais_altos:
    print(f"  {c['nome']}: {c['consumo']:.1f} kWh")

# 3. Lista de nomes em maiúsculo
nomes_upper = [c["nome"].upper() for c in consumidores]
print(f"\nNomes em maiúsculo (primeiros 3): {nomes_upper[:3]}")

# 4. Consumo total por classe (usando dict comprehension)
classes = set(c["classe"] for c in consumidores)
consumo_por_classe = {
    classe: sum(c["consumo"] for c in consumidores if c["classe"] == classe)
    for classe in classes
}

print("\nConsumo total por classe:")
for classe, total in sorted(consumo_por_classe.items(), key=lambda x: x[1], reverse=True):
    print(f"  {classe:<20}: {total:>10,.1f} kWh")

## 5. Prática: Classificar Consumidores por Faixa de Consumo

A ANEEL define faixas de consumo para fins de tributação e subsídio. Vamos implementar essa classificação.

In [ ]:
def classificar_faixa_consumo_residencial(consumo_kwh: float) -> dict:
    """
    Classifica um consumidor residencial nas faixas de consumo
    conforme critérios da ANEEL para subclasse Baixa Renda.
    
    Faixas (referência — regiões Sul/Sudeste/Centro-Oeste):
      Faixa 1: até 30 kWh
      Faixa 2: 31 a 100 kWh
      Faixa 3: 101 a 220 kWh
      Faixa 4: acima de 220 kWh (tarifa integral)
    """
    if consumo_kwh <= 30:
        faixa = "Faixa 1"
        descricao = "Até 30 kWh — desconto máximo"
        desconto_pct = 65
    elif consumo_kwh <= 100:
        faixa = "Faixa 2"
        descricao = "31 a 100 kWh — desconto intermediário"
        desconto_pct = 40
    elif consumo_kwh <= 220:
        faixa = "Faixa 3"
        descricao = "101 a 220 kWh — desconto reduzido"
        desconto_pct = 10
    else:
        faixa = "Faixa 4"
        descricao = "Acima de 220 kWh — tarifa integral"
        desconto_pct = 0
    
    return {
        "faixa": faixa,
        "descricao": descricao,
        "desconto_pct": desconto_pct,
        "consumo_kwh": consumo_kwh
    }

# Classificar todos os residenciais da base
residenciais = [c for c in consumidores if c["classe"] == "RESIDENCIAL"]

print("Classificação dos consumidores residenciais:\n")
print(f"{'Nome':<20} | {'Consumo':>10} | {'Faixa':<10} | {'Desconto':>10}")
print("-" * 60)

for c in residenciais:
    classificacao = classificar_faixa_consumo_residencial(c["consumo"])
    print(
        f"{c['nome']:<20} | "
        f"{c['consumo']:>8.1f} kWh | "
        f"{classificacao['faixa']:<10} | "
        f"{classificacao['desconto_pct']:>8}%"
    )

In [ ]:
# Resumo por faixa
from collections import Counter

# Classificar todos os consumidores
todos_classificados = []
for c in consumidores:
    if c["classe"] == "RESIDENCIAL":
        clf = classificar_faixa_consumo_residencial(c["consumo"])
        todos_classificados.append(clf["faixa"])
    else:
        todos_classificados.append("N/A (não residencial)")

contagem = Counter(todos_classificados)

print("Distribuição por faixa:")
for faixa, qtd in sorted(contagem.items()):
    print(f"  {faixa}: {qtd} consumidor(es)")